# Notebook Overview

This notebook provides an interactive, end-to-end exploration of the **key concepts behind preparing XRF detector-frame data for Noise2Noise denoising**. It focuses on understanding the data, how it is structured, how patches and batches are created, how augmentations work, and how standardization stabilizes training. All steps are illustrative: the notebook explains and visualizes the building blocks used in denoising pipelines.

### Main steps

1. **Data loading and inspection**  
   - Load per-detector elemental TIFF frames for a chosen element.  
   - Browse the full detector stack interactively to visualize contrast, noise levels, and detector-to-detector variability.

2. **Dataset preparation**  
   - Exclude undesired detector indices.  
   - Convert the remaining frames into a consistent `(N, H, W)` array.  
   - Load the corresponding weighted-sum image for reference.

3. **Patch tiling and overlap**  
   - Visualize how large detector images are decomposed into overlapping square patches.  
   - Inspect tile coverage, single-coverage vs. overlap regions, stride, and representative example patches.

4. **Noise2Noise sample construction**  
   - Demonstrate how input/target pairs are created via random splitting and averaging of detector frames.  
   - Show the parent full image, the crop region, and the resulting input/target patches.

5. **Spatial augmentations**  
   - Explore how rotations and flips are applied consistently to both input and target patches.  
   - Orientation markers help track transformations.

6. **Training-time concepts**  
   - Build intuition for batch size, patch size, steps per epoch, and samples per epoch using interactive controls.  
   - Optional benchmarking illustrates how these parameters affect throughput.

7. **Per-frame standardization**  
   - Apply and visualize the standardization used in denoising pipelines.  
   - Compare original and standardized frames and inspect their histograms with global contrast and binning.

---

## Copyright & License

**© 2025 Dmitry Karpov**  
Assistant Professor of Physics and Materials Science, Université Grenoble Alpes  
(Materials Modelling and Exploration Laboratory, MEM – UGA/CEA)

These notebooks were created as **educational material** for the  
**DIADEM Academy training series on U-Net and image analysis** (PEPR DIADEM)

Unless otherwise stated, all notebook content is distributed under the:

**Creative Commons Attribution–NonCommercial 4.0 International License (CC BY-NC 4.0)**  
https://creativecommons.org/licenses/by-nc/4.0/

You are free to **use, share, adapt, and modify** this material for **non-commercial research and teaching**,  
provided that **proper attribution** is given to the author.

Commercial use is **not permitted** without prior written permission.

### Please cite when using or adapting this material:
D. Karpov, *U-Net for Image Analysis — DIADEM Academy Training Materials*, Université Grenoble Alpes, 2025-present.

### Disclaimer
This material is provided *“as is”*, without warranty of any kind.  
The author and associated institutions are not liable for any damages arising from its use.

---

In [ ]:
# this line is just to check GPU status
!nvidia-smi

In [ ]:
# this cell is needed to mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from ipywidgets import (
    Button, IntSlider, HBox, VBox,
    HTML, Layout, Output, Dropdown, Checkbox
)

from IPython.display import display
from mpl_toolkits.axes_grid1 import make_axes_locatable

import tensorflow as tf
from tensorflow.keras import mixed_precision


# mixed precision for speed on Colab GPUs
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

In [ ]:
# set the path to the setkafluo library in the directory that
# you cloned with the notebooks
# Be careful! Sometimes Colab needs change of My Drive to MyDrive or vice versa
sys.path.append("/content/drive/MyDrive/setkafluo_demo/notebooks_and_library/libs/")

import setkafluo as skf
print("SetkaFluo version:", skf.__version__)

from setkafluo.denoise import (
    extract_covering_patches_with_overlap_pad,
    generate_noise2noise_samples,
    standardize_images,
    extract_random_patches,
    augment_patch,
)
from setkafluo.denoise_benchmark import clone_cfg, benchmark_steps

In [ ]:
# The path to the base folder
base_path = Path("/content/drive/MyDrive/setkafluo_demo/")
data_dir  = base_path / "input_data/" # path to the data


In [ ]:
# Pick the element you want to work with (e.g. "K-K", "Zn-K", "Ir-L", …)
element = "K-K"

def read_tif_files(directory_path, ardesia=True, element='Zn-K', elements_to_discard=None):
    """
    Returns:
        idxs (np.ndarray of int)
        images (list[np.ndarray])
        names (np.ndarray of str)
    """
    suffix = "_ardesia" if ardesia else "_arwen"
    records = []  # (idx, image, filename)

    for fname in os.listdir(directory_path):
        if fname.endswith(f"_{element}_area_density_ngmm2.tif") and "_element" in fname and suffix in fname:
            i0 = fname.index("_element") + len("_element")
            i1 = fname.index("_", i0)
            idx = int(fname[i0:i1])
            if elements_to_discard and idx in elements_to_discard:
                continue
            img = skf.load_tiff(os.path.join(directory_path, fname))
            records.append((idx, img, fname))

    # sort by index
    records.sort(key=lambda t: t[0])

    if not records:
        return np.array([], dtype=int), [], np.array([], dtype=str)

    idxs  = np.array([r[0] for r in records], dtype=int)
    imgs  = [r[1] for r in records]                 # list of np.ndarrays
    names = np.array([r[2] for r in records], dtype=str)
    return idxs, imgs, names


# Load Detector Frames + Interactive Stack Browser

**Purpose:** Read all detector-frame TIFFs for the chosen element and **interactively browse** them.

**What happens:**
- `read_tif_files(...)` scans the directory for files matching the element pattern (e.g., `*_elementXX_*_{element}_area_density_ngmm2.tif`), returning:
  - `idxs`: detector indices (sorted),
  - `images_stack`: list of 2D arrays,
  - `names`: file names.


In [ ]:
# Interactive visualization: click/slide through the stack with indices and filenames shown

# Load the full stack (no exclusions yet)
data_dir_stack = data_dir / "processed/separate_elements/"

idxs, images_stack, names = read_tif_files(str(data_dir_stack), ardesia=True, element=element)
print(f"Loaded {len(images_stack)} frames for {element}. Indexes: {idxs}")

if len(images_stack) == 0:
    print("No images found for the selected element.")
else:
    def to_numeric_img(x):
        arr = np.asarray(x)
        if arr.dtype == object:
            arr = arr.astype(np.float32)
        return arr

    def auto_contrast(img):
        p1, p99 = np.percentile(img, [1, 99])
        if p1 == p99:
            p1, p99 = float(np.min(img)), float(np.max(img))
        return p1, p99

    # Widgets
    slider = IntSlider(value=0, min=0, max=len(images_stack)-1, step=1,
                       description='Frame:', continuous_update=False)
    btn_prev = Button(description="⟵ Prev", layout=Layout(width="100px"))
    btn_next = Button(description="Next ⟶", layout=Layout(width="100px"))
    info = HTML()
    controls = HBox([btn_prev, slider, btn_next])
    ui = VBox([controls, info])

    out = Output()

    def draw(k: int):
        img = to_numeric_img(images_stack[k])
        vmin, vmax = auto_contrast(img)

        out.clear_output(wait=True)
        with out:
            # Larger figure so the image is bigger
            fig, ax = plt.subplots(figsize=(8.5, 8.5))
            im = ax.imshow(img, vmin=vmin, vmax=vmax)  # default colormap
            ax.set_title(f"{element}  •  detector index = {idxs[k]}", fontsize=12)
            ax.set_axis_off()

            # Colorbar with same height as the image axis
            divider = make_axes_locatable(ax)
            cax = divider.append_axes("right", size="3%", pad=0.05)
            cb = fig.colorbar(im, cax=cax)

            # Ensure colorbar height matches the image height exactly
            ax.set_box_aspect(img.shape[0] / img.shape[1])

            fig.tight_layout()
            plt.show()

        info.value = (
            f"<code>{k+1}/{len(images_stack)}</code> | "
            f"idx=<b>{idxs[k]}</b> | "
            f"min={np.min(img):.3g}, mean={np.mean(img):.3g}, max={np.max(img):.3g}"
        )

    def on_prev(_):
        if slider.value > slider.min:
            slider.value -= 1
            draw(slider.value)

    def on_next(_):
        if slider.value < slider.max:
            slider.value += 1
            draw(slider.value)

    def on_slide(change):
        if change["name"] == "value":
            draw(change["new"])

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    slider.observe(on_slide)

    display(ui, out)
    draw(slider.value)

# Exclude Bad Detectors & Stack to `(N, H, W)`

**Purpose:** Remove problematic detector indices and ensure the data is a single NumPy array suitable for training.

**What happens:**
- Set `bad_elements = [15]` (example) to exclude specific detectors.
- Reload a **filtered** `images_stack`.
- Convert the list to a dense array `(N, H, W)` with `np.stack(...)`.

In [ ]:
# After browsing above, set the indexes you want to exclude
bad_elements = [15]

# Reload the stack without the bad indexes
idxs, images_stack, names = read_tif_files(
    str(data_dir_stack), ardesia=True, element=element, elements_to_discard=bad_elements
)

print("Indexes:", idxs)
print("Files:", names)

# Ensure we have a proper NumPy stack for training: (N, H, W)
if isinstance(images_stack, list):
    try:
        images_stack = np.stack(images_stack, axis=0)
    except ValueError as e:
        raise ValueError(
            f"Could not stack images into a single array because shapes differ. "
            f"Please inspect and fix the inconsistent images. Original error: {e}"
        )

print("Stack shape:", images_stack.shape, images_stack.dtype)

# Load the Evaluation Image (`fluo1`) and Visualize

In [ ]:
# Evaluation (weighted sum) image
data_dir_weight = data_dir / "processed/weighted_sums/"

fluo1 = skf.load_tiff(
    data_dir_weight / f"IMG_fluo1_{element}_area_density_ngmm2.tif"
)
print("Eval image:", fluo1.shape, fluo1.dtype)

# Visualize with matched colorbar height
p1, p99 = np.percentile(fluo1, [1, 99])
if p1 == p99:
    p1, p99 = float(np.min(fluo1)), float(np.max(fluo1))

fig, ax = plt.subplots(figsize=(7.5, 7.5))
im = ax.imshow(fluo1, vmin=p1, vmax=p99)  # default colormap
ax.set_title(f"Weighted sum {element}")
ax.set_axis_off()

# Make colorbar axis with same height as image axis
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3%", pad=0.05)
cb = fig.colorbar(im, cax=cax)

# Preserve image aspect so the colorbar height matches
ax.set_box_aspect(fluo1.shape[0] / fluo1.shape[1])

fig.tight_layout()
plt.show()

# How PATCH and overlap tile your image

This demo shows how a single large image is decomposed into smaller square tiles (PATCH × PATCH) for training and inference, and where those tiles overlap.

## What you’re looking at
- **Gray image:** the selected detector frame.
- **White boxes:** the tiling grid (one box per tile).
- **Blue overlay:** pixels covered by **exactly one** tile.
- **Red overlay:** pixels covered by **two or more** tiles (overlap regions).
- **Cyan box:** example tile #0.
- **Yellow box:** the tile that covers the image center.

## Why overlaps?
- Overlap reduces edge artifacts during prediction (we blend overlapping tiles).
- A larger **overlap** → more blending and smoother seams; smaller overlap → faster but riskier edges.

## Controls
- **PATCH:** tile size in pixels. (Cannot exceed the image size.)
- **overlap:** how much neighboring tiles overlap. (Automatically limited to PATCH−1.)

**Tip:** Try increasing PATCH while keeping overlap roughly 1/2 PATCH to see how coverage changes.

In [ ]:
# PATCH demo: gray image + blue/red coverage overlays; highlight two example patches
# pick a demo frame
_demo_img = images_stack[0] if isinstance(images_stack, (list, tuple)) else images_stack[0]
_demo_img = np.asarray(_demo_img, dtype=float)
H, W = _demo_img.shape[:2]

def _auto_limits(x):
    p1, p99 = np.percentile(x, [1, 99])
    if p1 == p99:
        p1, p99 = float(np.min(x)), float(np.max(x))
    return p1, p99

# PATCH can't exceed the image side; start with a sensible default
max_patch = int(min(min(H, W), 256))
patch_slider   = IntSlider(value=min(32, max_patch), min=16, max=max_patch, step=16,
                           description="PATCH", continuous_update=False)
overlap_slider = IntSlider(value=16, min=0, max=max(0, patch_slider.value - 1), step=8,
                           description="overlap", continuous_update=False)
info = HTML()
out  = Output()

def _draw_patch_grid(patch: int, overlap: int):
    stride = max(1, patch - overlap)
    patches, coords = extract_covering_patches_with_overlap_pad(_demo_img, patch, overlap)

    # Coverage map: how many tiles cover each pixel (inside image bounds)
    cover = np.zeros((H, W), dtype=np.uint16)
    for (top, left) in coords:
        y0, x0 = max(top, 0), max(left, 0)
        y1, x1 = min(top + patch, H), min(left + patch, W)
        if y0 < y1 and x0 < x1:
            cover[y0:y1, x0:x1] += 1

    single_mask  = (cover == 1)   # blue = exactly one tile covers the pixel
    overlap_mask = (cover >= 2)   # red  = two or more tiles cover the pixel

    # Find a patch that covers the image center (or the closest one)
    cy, cx = H // 2, W // 2
    center_idx = None
    for i, (top, left) in enumerate(coords):
        if (top <= cy < top + patch) and (left <= cx < left + patch):
            center_idx = i
            break
    if center_idx is None:
        center_idx = int(np.argmin([
            ( (top + patch/2 - cy)**2 + (left + patch/2 - cx)**2 )
            for (top, left) in coords
        ]))

    out.clear_output(wait=True)
    with out:
        # MAIN FIGURE: gray image + tile boxes + blue single-cover + red overlap (no colorbar)
        fig, ax = plt.subplots(figsize=(9.0, 9.0))
        vmin, vmax = _auto_limits(_demo_img)
        im = ax.imshow(_demo_img, vmin=vmin, vmax=vmax, cmap='gray')
        ax.set_title(
            f"{element}: PATCH={patch}, overlap={overlap}, stride={stride}px • tiles={len(coords)}",
            fontsize=12
        )
        ax.set_axis_off()

        # Light rectangles for all tiles
        for (top, left) in coords:
            rect = Rectangle((left, top), patch, patch, fill=False, linewidth=0.8, alpha=0.35, color='white')
            ax.add_patch(rect)

        # Blue overlay for single-cover pixels (very transparent)
        blue = np.zeros((H, W, 4), dtype=float)
        blue[..., 2] = 1.0                         # B
        blue[..., 3] = single_mask.astype(float) * 0.12
        ax.imshow(blue, interpolation="nearest")

        # Red overlay for overlapping pixels (very transparent, drawn last)
        red = np.zeros((H, W, 4), dtype=float)
        red[..., 0] = 1.0                           # R
        red[..., 3] = overlap_mask.astype(float) * 0.12
        ax.imshow(red, interpolation="nearest")

        # Highlight the two example patches with thicker outlines
        ex0_top, ex0_left = coords[0]
        cen_top, cen_left = coords[center_idx]
        ax.add_patch(Rectangle((ex0_left, ex0_top), patch, patch, fill=False,
                               linewidth=2.5, color='#00FFFF', alpha=0.9))  # cyan
        ax.add_patch(Rectangle((cen_left, cen_top), patch, patch, fill=False,
                               linewidth=2.5, color='#FFD400', alpha=0.9))  # yellow

        ax.set_box_aspect(H / W)
        fig.tight_layout()
        plt.show()

        # SECOND FIGURE: the two example extracted patches (matching border colors)
        fig2, axs = plt.subplots(1, 2, figsize=(9.0, 4.5))

        def _style_axes(ax, color):
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(2.5)

        # Example #0 (cyan)
        p1 = patches[0]
        p1v = np.percentile(p1, [1, 99])
        axs[0].imshow(p1, vmin=p1v[0], vmax=p1v[1], cmap='gray')
        axs[0].set_title("Example patch #0", fontsize=11)
        axs[0].set_axis_off()
        _style_axes(axs[0], '#00FFFF')

        # Center-covering (yellow)
        p2 = patches[center_idx]
        p2v = np.percentile(p2, [1, 99])
        axs[1].imshow(p2, vmin=p2v[0], vmax=p2v[1], cmap='gray')
        axs[1].set_title("Patch covering image center", fontsize=11)
        axs[1].set_axis_off()
        _style_axes(axs[1], '#FFD400')

        fig2.tight_layout()
        plt.show()

    # grid summary / legend
    ys = sorted({y for y, _ in coords})
    xs = sorted({x for _, x in coords})
    info.value = (
        f"<b>Grid:</b> {len(ys)} rows × {len(xs)} cols &nbsp;|&nbsp; "
        f"stride = {stride}px &nbsp;|&nbsp; "
        f"<span style='color:#00FFFF'>cyan</span> = Example #0, "
        f"<span style='color:#FFD400'>yellow</span> = Center patch, "
        f"<span style='color:#1f77b4'>blue</span> = single cover, "
        f"<span style='color:#d62728'>red</span> = overlaps"
    )

def _on_change(_):
    # keep overlap <= PATCH-1 and update its max when PATCH changes
    p = int(patch_slider.value)
    new_max = max(0, p - 1)
    if overlap_slider.max != new_max:
        overlap_slider.max = new_max
    o = min(int(overlap_slider.value), new_max)
    if o != overlap_slider.value:
        overlap_slider.value = o
    _draw_patch_grid(p, o)

patch_slider.observe(_on_change, "value")
overlap_slider.observe(_on_change, "value")

display(VBox([HBox([patch_slider, overlap_slider]), info, out]))
_draw_patch_grid(patch_slider.value, overlap_slider.value)

# What "batch" means (and where each patch comes from)

This demo connects the abstract idea of a **batch** to concrete image regions.

## What you’re looking at
- **Left figure:** one "parent" standardized image (the input side) with a **yellow rectangle** showing **where the current patch was cropped**.
- **Right pair:** the **input patch** and the **target patch** for that sample.

## How the input/target are formed (Noise2Noise)
- From the full stack of detector frames, we randomly split all frames into two disjoint halves.
- **Input = average of one half**, **Target = average of the remaining half**.
- Crops of size `DEMO_PATCH × DEMO_PATCH` are then extracted at the **same location** from both averages.

## Why batches?
- Each **batch** is a small collection of such (input, target) pairs processed together in one training **step**.
- Larger `batch_size` → more examples per step (smoother gradients) but higher memory use.
- The **Reshuffle** button rebuilds the batch with new random pairs and new crop locations.

In [ ]:
# BATCH demo: full image + patch rectangle + patch pair
DEMO_PATCH = 64
DEMO_BATCH = 10
CMAP = 'viridis'  # <- change here if you want e.g. 'plasma', 'magma', 'cividis', etc.

# Ensure (N, H, W)
_stack = images_stack if isinstance(images_stack, np.ndarray) else np.stack(images_stack, axis=0)

def make_one_batch_with_coords(batch_size=DEMO_BATCH, patch=DEMO_PATCH):
    """Create a demo batch and remember crop coords for each sample."""
    xin, xgt = generate_noise2noise_samples(_stack, n_samples=batch_size)
    xin_std, _, _ = standardize_images(xin)
    xgt_std, _, _ = standardize_images(xgt)

    H, W = xin_std.shape[1], xin_std.shape[2]
    pin, pgt, coords = [], [], []
    rng = np.random.default_rng()

    for i in range(batch_size):
        top  = int(rng.integers(0, H - DEMO_PATCH + 1))
        left = int(rng.integers(0, W - DEMO_PATCH + 1))
        pi = xin_std[i, top:top+DEMO_PATCH, left:left+DEMO_PATCH][..., None]
        pg = xgt_std[i, top:top+DEMO_PATCH, left:left+DEMO_PATCH][..., None]
        pin.append(pi); pgt.append(pg); coords.append((top, left))

    return xin_std, xgt_std, np.asarray(pin), np.asarray(pgt), coords

# Build initial batch
xin_std, xgt_std, pin, pgt, coords = make_one_batch_with_coords()

# UI
idx       = IntSlider(value=0, min=0, max=pin.shape[0]-1, step=1, description="sample", continuous_update=False)
prev_btn  = Button(description="⟵ Prev", layout=Layout(width="96px"))
next_btn  = Button(description="Next ⟶", layout=Layout(width="96px"))
reshuffle = Button(description="↻ Reshuffle", layout=Layout(width="120px"))
info      = HTML()
out       = Output()

# Clean any old observers if cell is re-run
try: idx.unobserve_all()
except Exception: pass
for b in (prev_btn, next_btn, reshuffle):
    try: b._click_handlers.callbacks.clear()
    except Exception: pass

def _percentiles(a):
    p1, p99 = np.percentile(a, [1, 99])
    if p1 == p99:
        p1, p99 = float(np.min(a)), float(np.max(a))
    return p1, p99

def draw(k: int):
    out.clear_output(wait=True)
    with out:
        # --- full parent image (input side) with rectangle ---
        fig1, ax1 = plt.subplots(figsize=(8.5, 8.5))
        vmin, vmax = _percentiles(xin_std[k])
        ax1.imshow(xin_std[k], vmin=vmin, vmax=vmax, cmap=CMAP)
        ax1.set_title(f"Parent input image — index={k} ({k+1}/{pin.shape[0]})", fontsize=12)
        ax1.set_axis_off()
        top, left = coords[k]
        rect = plt.Rectangle((left, top), DEMO_PATCH, DEMO_PATCH,
                             fill=False, linewidth=2.8, edgecolor='yellow')
        ax1.add_patch(rect)
        ax1.set_box_aspect(xin_std[k].shape[0] / xin_std[k].shape[1])
        fig1.tight_layout()
        plt.show()

        # --- the corresponding input/target patches ---
        fig2, axs = plt.subplots(1, 2, figsize=(8.5, 4.25))
        v1 = _percentiles(pin[k, ..., 0])
        v2 = _percentiles(pgt[k, ..., 0])
        axs[0].imshow(pin[k, ..., 0], vmin=v1[0], vmax=v1[1], cmap=CMAP)
        axs[1].imshow(pgt[k, ..., 0], vmin=v2[0], vmax=v2[1], cmap=CMAP)
        axs[0].set_title("input patch"); axs[1].set_title("target patch")
        for a in axs: a.set_axis_off()
        fig2.tight_layout()
        plt.show()

    info.value = (
        f"<b>Index:</b> {k} &nbsp;|&nbsp; "
        f"<b>Patch coords:</b> top={top}, left={left} &nbsp;|&nbsp; "
        f"size={DEMO_PATCH}×{DEMO_PATCH} &nbsp;|&nbsp; "
        f"<i>(Use slider / Prev / Next; ‘Reshuffle’ makes a new random batch.)</i>"
    )

def _on_slider(change):
    if change.get("name") != "value":
        return
    draw(int(change["new"]))

def _on_prev(_):
    if idx.value > idx.min:
        idx.value = idx.value - 1

def _on_next(_):
    if idx.value < idx.max:
        idx.value = idx.value + 1

def _on_reshuffle(_):
    global xin_std, xgt_std, pin, pgt, coords
    xin_std, xgt_std, pin, pgt, coords = make_one_batch_with_coords()
    idx.max = pin.shape[0] - 1
    idx.value = 0  # triggers draw(0)

# Wire events
idx.observe(_on_slider, names="value")
prev_btn.on_click(_on_prev)
next_btn.on_click(_on_next)
reshuffle.on_click(_on_reshuffle)

display(VBox([HBox([idx, prev_btn, next_btn, reshuffle]), info, out]))
draw(idx.value)

# Augmentations: before vs after (and how we keep orientation straight)

Training uses simple spatial augmentations to make the model invariant to rotations and flips. This demo shows each patch **before** and **after** the same augmentation on both input and target.

## What you’re looking at (2×2 layout)
- **Left column:** input patch (top = before, bottom = after augmentation).
- **Right column:** target patch (top = before, bottom = after augmentation).
- Colored corner markers (red = near top-left, blue = near top-right, yellow = near bottom-left) **move with the image** so you can see exactly how the augmentation transforms orientation.

## Which augmentations?
- Randomly chosen per sample from: **none**, **rotate 90°/180°/270°**, **flip left-right**, **flip up-down**.
- The same transform is applied to both input and target to keep them aligned.

## Why augment?
- Improves generalization and reduces overfitting by exposing the model to varied orientations and symmetries present in XRF data.
- Crucially, augmentations preserve the underlying signal while changing appearance.

In [ ]:
# Augmentation demo: before vs after, markers follow the augmentation
# Colormap (fallback if not defined earlier)
CMAP = globals().get("CMAP", "viridis")

DEMO_PATCH = 64
DEMO_BATCH = 12

# Ensure (N,H,W) stack
_stack = images_stack if isinstance(images_stack, np.ndarray) else np.stack(images_stack, axis=0)

# --- helpers to apply + track the same augmentation ---
def _apply_random_aug_with_meta(a_in, a_gt):
    """
    Mimic training augmentations while returning the transform meta so we
    can map marker coordinates consistently.
      opt=0: none
      opt=1: rot90 k (k in {1,2,3})
      opt=2: fliplr
      opt=3: flipud
    """
    opt = np.random.randint(0, 4)
    meta = {"opt": opt, "k": None}

    if opt == 0:
        return a_in, a_gt, meta
    if opt == 1:
        k = int(np.random.randint(1, 4))
        meta["k"] = k
        return np.rot90(a_in, k), np.rot90(a_gt, k), meta
    if opt == 2:
        return np.fliplr(a_in), np.fliplr(a_gt), meta
    # opt == 3
    return np.flipud(a_in), np.flipud(a_gt), meta

def _transform_points(pts, shape, meta):
    """
    Transform a list of (y,x) points according to 'meta' for an array of size 'shape' (H,W).
    Uses the same conventions as numpy rot90/flips applied above.
    """
    H, W = shape
    opt = meta["opt"]
    k   = meta.get("k", 0)

    out = []
    for (y, x) in pts:
        if opt == 0:     # none
            yy, xx = y, x
        elif opt == 1:   # rot90 k CCW
            if k % 4 == 1:   yy, xx = (W - 1 - x, y)
            elif k % 4 == 2: yy, xx = (H - 1 - y, W - 1 - x)
            elif k % 4 == 3: yy, xx = (x, H - 1 - y)
            else:            yy, xx = y, x
        elif opt == 2:   # fliplr
            yy, xx = (y, W - 1 - x)
        else:            # flipud
            yy, xx = (H - 1 - y, x)
        out.append((yy, xx))
    return out

def _base_markers(shape, inset_frac=0.04):
    """Three base points near TL/TR/BL so they remain visible; returns list[(y,x)], colors."""
    H, W = shape
    dy = max(1, int(round(H * inset_frac)))
    dx = max(1, int(round(W * inset_frac)))
    pts = [
        (           dy,            dx),   # near top-left
        (           dy, W - 1 - dx),      # near top-right
        (H - 1 - dy,            dx),      # near bottom-left
    ]
    cols = ["red", "blue", "yellow"]
    return pts, cols

def _marker_radius(shape):
    # scale marker radius with image size (gentle scaling)
    H, W = shape
    return max(5.0, min(H, W) * 0.02)

def _draw_markers(ax, pts, cols, shape):
    r = _marker_radius(shape)
    for (y, x), c in zip(pts, cols):
        circ = plt.Circle((x, y), radius=r, facecolor=c, edgecolor="black", linewidth=1.5, alpha=0.9)
        ax.add_patch(circ)

# --- build an augmented batch with per-sample meta so markers can follow ---
def make_aug_batch_with_meta(batch_size=DEMO_BATCH, patch=DEMO_PATCH):
    xin, xgt = generate_noise2noise_samples(_stack, n_samples=batch_size)
    xin_std, _, _ = standardize_images(xin)
    xgt_std, _, _ = standardize_images(xgt)
    pin, pgt = extract_random_patches(xin_std, xgt_std, patch)  # (B,P,P,1)

    aug_in, aug_gt, metas = [], [], []
    for i in range(pin.shape[0]):
        ai, ag, meta = _apply_random_aug_with_meta(pin[i, ..., 0], pgt[i, ..., 0])  # drop channel for ops
        aug_in.append(ai[..., None]); aug_gt.append(ag[..., None]); metas.append(meta)
    return pin, pgt, np.asarray(aug_in), np.asarray(aug_gt), metas

# Build initial data
pin, pgt, aug_in, aug_gt, metas = make_aug_batch_with_meta()

# Widgets
idx       = IntSlider(value=0, min=0, max=pin.shape[0]-1, description="sample", continuous_update=False)
btn_prev  = Button(description="⟵ Prev", layout=Layout(width="90px"))
btn_next  = Button(description="Next ⟶", layout=Layout(width="90px"))
reshuffle = Button(description="↻ Reshuffle", layout=Layout(width="110px"))
info      = HTML()
out       = Output()

# Clean old observers/handlers if re-run
try: idx.unobserve_all()
except Exception: pass
for b in (btn_prev, btn_next, reshuffle):
    try: b._click_handlers.callbacks.clear()
    except Exception: pass

def _pcts(a):
    p1, p99 = np.percentile(a, [1, 99])
    if p1 == p99:
        p1, p99 = float(np.min(a)), float(np.max(a))
    return p1, p99

def draw_aug(k: int):
    out.clear_output(wait=True)
    with out:
        # 2x3 grid with narrow center spacer for clearer L vs R separation
        fig = plt.figure(figsize=(10.5, 9.2), constrained_layout=True)
        gs  = fig.add_gridspec(2, 3, width_ratios=[1, 0.06, 1], height_ratios=[1, 1])

        ax_in_before = fig.add_subplot(gs[0, 0])
        ax_in_after  = fig.add_subplot(gs[1, 0])
        ax_tg_before = fig.add_subplot(gs[0, 2])
        ax_tg_after  = fig.add_subplot(gs[1, 2])

        # hide spacer column axes
        for r in (0, 1):
            ax_sp = fig.add_subplot(gs[r, 1]); ax_sp.set_visible(False)

        # Base markers for this patch (before)
        H, W = pin.shape[1], pin.shape[2]
        base_pts, base_cols = _base_markers((H, W))
        # Transform markers by this sample's augmentation meta
        aug_pts = _transform_points(base_pts, (H, W), metas[k])

        def show(ax, img, title, markers_pts):
            vmin, vmax = _pcts(img)
            ax.imshow(img, vmin=vmin, vmax=vmax, cmap=CMAP)
            _draw_markers(ax, markers_pts, base_cols, img.shape)
            ax.set_title(title); ax.set_axis_off()

        # Left column: input before/after with markers tracked
        show(ax_in_before, pin[k,  ..., 0], "input (before)", base_pts)
        show(ax_in_after,  aug_in[k, ..., 0], "input (after aug)", aug_pts)

        # Right column: target before/after with same transform
        show(ax_tg_before, pgt[k,  ..., 0], "target (before)", base_pts)
        show(ax_tg_after,  aug_gt[k, ..., 0], "target (after aug)", aug_pts)

        plt.show()

    meta = metas[k]
    mtxt = "none" if meta["opt"] == 0 else ("rot90 k=%d" % meta["k"] if meta["opt"] == 1 else ("fliplr" if meta["opt"] == 2 else "flipud"))
    info.value = (
        f"<b>Sample:</b> {k+1}/{pin.shape[0]} &nbsp;|&nbsp; "
        f"patch = {pin.shape[1]}×{pin.shape[2]} &nbsp;|&nbsp; "
        f"<b>aug:</b> {mtxt} &nbsp;|&nbsp; "
        f"<i>Use slider or buttons; ‘Reshuffle’ builds a new random batch & augmentations.</i>"
    )

def on_idx(change):
    if change.get("name") == "value":
        draw_aug(int(change["new"]))

def on_prev(_):
    if idx.value > idx.min:
        idx.value -= 1

def on_next(_):
    if idx.value < idx.max:
        idx.value += 1

def on_reshuffle(_):
    global pin, pgt, aug_in, aug_gt, metas
    pin, pgt, aug_in, aug_gt, metas = make_aug_batch_with_meta()
    idx.max = pin.shape[0] - 1
    idx.value = 0  # triggers draw

# Wire up
idx.observe(on_idx, names="value")
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
reshuffle.on_click(on_reshuffle)

controls = HBox([idx, btn_prev, btn_next, reshuffle], layout=Layout(align_items="center"))
display(VBox([controls, info, out]))
draw_aug(idx.value)

# What does `steps_per_epoch` mean?

Our pipeline is an **infinite generator** (it keeps creating fresh Noise2Noise pairs and random patches). That changes what an “epoch” means:

- **Batch**  
  A small set of samples processed together (here: random **patch pairs**). Size = `batch_size`.

- **Step**  
  One forward + backward pass **on one batch** → **one optimizer update**.

- **steps_per_epoch**  
  How many steps/updates we do per epoch (so we can log metrics, run callbacks, save checkpoints, etc.).

- **Epoch**  
  In our setup, **not** a full pass over a fixed dataset. It’s simply **`steps_per_epoch` updates** grouped together.

## Quick math

- **Samples per epoch** (here, number of patches seen per epoch):  
  `samples_per_epoch ≈ batch_size × steps_per_epoch`

- **Time per epoch** grows roughly linearly with `steps_per_epoch`.

## Why adjust `steps_per_epoch`?

- **Higher** `steps_per_epoch` → model sees **more distinct random patches** each epoch → steadier metrics, but **longer epochs**.
- **Lower** `steps_per_epoch` → faster epochs & quicker feedback, but noisier metrics/updates.

## Practical tips

- With small batches (2–8), start with **64–256 steps/epoch**.
- If loss/PSNR is very jittery → **increase** `steps_per_epoch` (or `batch_size`).
- If each epoch takes too long → **decrease** `steps_per_epoch` and/or save checkpoints more frequently.

In [ ]:
# STEPS demo: interactive intuition + real-model benchmark
# Sliders
batch_slider = IntSlider(value=4, min=1, max=16, step=1,
                         description="batch_size", continuous_update=False)
steps_slider = IntSlider(value=128, min=16, max=512, step=16,
                         description="steps/epoch", continuous_update=False)
patch_slider = IntSlider(value=int(globals().get("PATCH", 32)), min=16, max=128, step=16,
                         description="patch", continuous_update=False)

# Presets (batch, steps, patch)
presets = Dropdown(
    options=[
        ("Fast feedback",            (4,  64,  32)),
        ("Balanced (default)",       (4, 128,  32)),
        ("Thorough (longer epoch)",  (4, 256,  32)),
        ("Bigger batch",             (8, 128,  32)),
        ("Bigger batch + longer",    (8, 256,  32)),
        ("Larger patch",             (4, 128,  64)),
    ],
    value=(4, 128, 32),
    description="Presets",
    layout=Layout(width="300px")
)

# If the real pipeline is possible (images_stack exists), default to True
_use_real_possible = "images_stack" in globals()
use_real_cb = Checkbox(value=_use_real_possible, description="Use real pipeline", indent=False,
                       layout=Layout(width="170px"))
if not _use_real_possible:
    use_real_cb.disabled = True
    use_real_cb.description = "Use real pipeline (unavailable)"

btn_apply = Button(description="Apply preset", layout=Layout(width="130px"))
btn_bench = Button(description="Benchmark 30 steps", layout=Layout(width="180px"))

info = HTML()
out  = Output()

# Clean old handlers if re-run
for b in (btn_apply, btn_bench):
    try: b._click_handlers.callbacks.clear()
    except Exception: pass
for sld in (batch_slider, steps_slider, patch_slider):
    try: sld.unobserve_all()
    except Exception: pass

def samples_per_epoch(batch_size, steps_per_epoch):
    return int(batch_size) * int(steps_per_epoch)

def _fmt_time(seconds: float) -> str:
    if not np.isfinite(seconds) or seconds <= 0:
        return "n/a"
    m, s = divmod(seconds, 60)
    return f"{int(m)}m {s:0.1f}s" if m >= 1 else f"{s:0.1f}s"

bench_text = ""

def _stack_array(maybe_list_or_array):
    if isinstance(maybe_list_or_array, np.ndarray):
        return maybe_list_or_array
    if isinstance(maybe_list_or_array, (list, tuple)):
        return np.stack(maybe_list_or_array, axis=0)
    raise ValueError("images_stack must be a list of arrays or an ndarray.")

def redraw():
    b = int(batch_slider.value)
    s = int(steps_slider.value)
    candidates = [32, 64, 128, 256, 512]
    counts = [samples_per_epoch(b, c) for c in candidates]

    out.clear_output(wait=True)
    with out:
        fig, ax = plt.subplots(figsize=(7.6, 3.9))
        bars = ax.bar([str(c) for c in candidates], counts)
        ax.set_xlabel("steps_per_epoch")
        ax.set_ylabel("samples / epoch  (approx. patch pairs)")
        ax.set_title(f"batch_size = {b}  ->  samples/epoch = batch_size x steps_per_epoch")
        for rect, val in zip(bars, counts):
            ax.text(rect.get_x()+rect.get_width()/2, rect.get_height(),
                    f"{val:,}", ha="center", va="bottom", fontsize=9)
        fig.tight_layout()
        plt.show()

    base = (
        f"<b>Current:</b> batch_size = <code>{b}</code>, "
        f"steps_per_epoch = <code>{s}</code>, patch = <code>{int(patch_slider.value)}</code> "
        f"-> <b>{samples_per_epoch(b, s):,}</b> samples/epoch.<br>"
        f"<i>One 'step' = one optimizer update on one batch. "
        f"An 'epoch' here is just a group of steps_per_epoch steps (the dataset can be infinite).</i>"
    )
    info.value = base + (("<br>" + bench_text) if bench_text else "")

def on_preset(_):
    b, s, p = presets.value
    batch_slider.value = b
    steps_slider.value = s
    patch_slider.value = p
    redraw()

def on_benchmark(_):
    global bench_text
    bench_text = ""

    b = int(batch_slider.value)
    s = int(steps_slider.value)
    p = int(patch_slider.value)

    base_cfg = globals().get("cfg", skf.DenoiseConfig(
        lr=1e-5, batch_size=b, patch_size=p, steps_per_epoch=s, epochs=1,
        save_models_dir=None, mixed_precision_warn=False
    ))
    tmp_cfg = clone_cfg(base_cfg, batch_size=b, patch_size=p, steps_per_epoch=s)

    use_real = bool(use_real_cb.value and _use_real_possible)
    stack_arr = None
    if use_real:
        try:
            stack_arr = _stack_array(globals()["images_stack"])
        except Exception as e:
            use_real = False
            print(f"[note] Falling back to synthetic pipeline: {type(e).__name__}: {e}")

    mdl = globals().get("model", None)

    out.clear_output(wait=True)
    with out:
        print(f"Benchmarking 30 training steps "
              f"(batch={b}, patch={p}, pipeline={'real' if use_real else 'synthetic'}) ...")

    try:
        res = benchmark_steps(
            tmp_cfg,
            steps=30,
            warmup=5,
            model=mdl,
            use_real_pipeline=use_real,
            stack=stack_arr
        )
        sps = res["steps_per_second"]
        sec_per_epoch = res["seconds_per_epoch_est"]
        bench_text = (
            f"Estimated time/epoch ~ <b>{_fmt_time(sec_per_epoch)}</b> "
            f"({sps:.2f} steps/s, pipeline = {res['used_pipeline']})."
        )
    except Exception as e:
        bench_text = f"<span style='color:#b00'>Benchmark failed:</span> {type(e).__name__}: {e}"
    finally:
        redraw()

def on_change(_):
    global bench_text
    bench_text = ""
    redraw()

# Wire up
btn_apply.on_click(on_preset)
btn_bench.on_click(on_benchmark)
batch_slider.observe(on_change, names="value")
steps_slider.observe(on_change, names="value")
patch_slider.observe(on_change, names="value")

# Compact control bar
controls_left  = VBox([batch_slider, steps_slider, patch_slider],
                      layout=Layout(align_items="flex-start"))
controls_right = VBox(
    [presets, HBox([use_real_cb, btn_apply, btn_bench], layout=Layout(gap="8px"))],
    layout=Layout(align_items="flex-start", justify_content="center")
)
controls = HBox([controls_left, controls_right],
                layout=Layout(align_items="center", justify_content="space-between", width="900px"))

display(VBox([controls, info, out]))
redraw()

# Why we standardize each frame (and how to read the plots)

**Goal:** make the learning problem numerically stable and comparable across detectors/samples whose absolute intensities vary.

## Per-frame standardization used in this notebook
- For each 2D image `img`, compute its mean `μ = mean(img)`.
- Remove global scale: `z = img / μ - 1`.
- Normalize contrast: `s = z / σ`, where `σ = std(z)`.
- A tiny epsilon `ε` is used internally to avoid division by zero when `μ≈0` or `σ≈0`.

So the model sees inputs with ~0 mean and ~1 std:

**Forward (to training domain)**  
`s = (img / μ - 1) / σ`

**Inverse (back to physical units)**  
`img ≈ (s * σ + 1) * μ`

## What the figure shows
- **Left column — images (fixed global contrast):**
  - **Top:** original detector frame (all detectors share the same 1st–99th percentile stretch computed globally over the stack).
  - **Bottom:** standardized frame `s` (again using a global 1st–99th percentile stretch, but in standardized units).
  - Using fixed global contrast makes inter-detector brightness differences visible and comparable.

- **Right column — separate histograms (fixed global bins):**
  - **Top:** histogram in **original units** `x` with vertical markers at `μ`, `μ(1−σ)`, and `μ(1+σ)`.
  - **Bottom:** histogram in **standardized units** `s` with reference lines at `s = −1, 0, +1`.
  - Bin edges/ranges are fixed globally (across all detectors) so shapes remain comparable as you scroll.

## Why this helps
- Puts all patches on a common numeric scale → more stable loss/gradients (especially with mixed precision).
- Makes MSE/PSNR magnitudes more comparable across steps/epochs.
- Encourages the network to learn **structure** rather than absolute magnitude; we restore physical units **after** prediction via the inverse formula.

## Notes
- If a frame is almost constant, `σ` can be tiny; `ε` prevents numerical blow-ups, but such frames carry little learning signal.
- Percentile stretching and histogram binning are **for visualization only**; the training pipeline does not clip your data.
- At inference: standardize input → predict → **undo** standardization to return to ng/mm² (or your native units).

In [ ]:
# Standardization across detectors: images + separate histograms (original vs standardized)
# Colormap (fallback if not set earlier)
CMAP = globals().get("CMAP", "viridis")

# Ensure (N, H, W)
_stack = images_stack if isinstance(images_stack, np.ndarray) else np.stack(images_stack, axis=0)
N, H, W = _stack.shape

# Standardize once for whole stack
std_stack, mus, sigmas = standardize_images(_stack)  # (N,H,W), per-detector μ and σ (of z = x/μ - 1)

# Global contrast ranges (1–99%) across ALL detectors
def _pcts(arr, p=(1, 99)):
    p1, p99 = np.percentile(arr, list(p))
    if p1 == p99:
        p1, p99 = float(np.min(arr)), float(np.max(arr))
    return p1, p99

vmin_orig, vmax_orig = _pcts(_stack)     # originals
vmin_std,  vmax_std  = _pcts(std_stack)  # standardized

# Histogram bins fixed globally (makes shapes comparable across detectors)
hist_bins = 200
orig_bins = np.linspace(vmin_orig, vmax_orig, hist_bins)
std_bins  = np.linspace(vmin_std,  vmax_std,  hist_bins)

# Widgets
idx       = IntSlider(value=0, min=0, max=N-1, step=1, description="detector", continuous_update=False)
btn_prev  = Button(description="⟵ Prev", layout=Layout(width="90px"))
btn_next  = Button(description="Next ⟶", layout=Layout(width="90px"))
info      = HTML()
out       = Output()

# Clean old observers if re-run
try: idx.unobserve_all()
except Exception: pass
for b in (btn_prev, btn_next):
    try: b._click_handlers.callbacks.clear()
    except Exception: pass

def draw(k: int):
    out.clear_output(wait=True)
    img_orig = _stack[k]
    img_std  = std_stack[k]
    mu  = float(mus[k])
    sig = float(sigmas[k])

    with out:
        # Grid: left column images; right column separate histograms
        fig = plt.figure(figsize=(12.5, 8.6), constrained_layout=True)
        # Add a bit more padding to avoid any overlaps with the suptitle
        fig.set_constrained_layout_pads(w_pad=0.03, h_pad=0.08, wspace=0.02, hspace=0.02)

        gs  = fig.add_gridspec(2, 2, width_ratios=[1.08, 0.9], height_ratios=[1, 1])

        ax_img_orig = fig.add_subplot(gs[0, 0])
        ax_img_std  = fig.add_subplot(gs[1, 0])
        ax_hist_x   = fig.add_subplot(gs[0, 1])
        ax_hist_s   = fig.add_subplot(gs[1, 1])

        # --- Images with global contrast ---
        ax_img_orig.imshow(img_orig, vmin=vmin_orig, vmax=vmax_orig, cmap=CMAP)
        det_label = f"{idxs[k]}" if 'idxs' in globals() else str(k)
        ax_img_orig.set_title(f"Original (detector idx = {det_label})", pad=6)
        ax_img_orig.set_axis_off()
        ax_img_orig.set_box_aspect(H / W)

        ax_img_std.imshow(img_std, vmin=vmin_std, vmax=vmax_std, cmap=CMAP)
        ax_img_std.set_title("Standardized", pad=6)
        ax_img_std.set_axis_off()
        ax_img_std.set_box_aspect(H / W)

        # --- Histogram: original domain (x) ---
        x_vals = img_orig.ravel()
        ax_hist_x.hist(x_vals, bins=orig_bins, density=True, color="tab:orange", alpha=0.85)
        ax_hist_x.set_xlim(orig_bins[0], orig_bins[-1])
        ax_hist_x.set_xlabel("original units (x)")
        ax_hist_x.set_ylabel("density (original)")
        ax_hist_x.grid(True, alpha=0.2)
        ax_hist_x.axvline(mu, color="k", ls="--", lw=1.0, alpha=0.7)
        ax_hist_x.text(mu, ax_hist_x.get_ylim()[1]*0.95, "μ", ha="center", va="top", fontsize=9)
        ax_hist_x.axvline(mu*(1+sig), color="k", ls=":", lw=1.0, alpha=0.7)
        ax_hist_x.text(mu*(1+sig), ax_hist_x.get_ylim()[1]*0.95, "μ(1+σ)", ha="center", va="top", fontsize=9)
        ax_hist_x.axvline(mu*(1-sig), color="k", ls=":", lw=1.0, alpha=0.7)
        ax_hist_x.text(mu*(1-sig), ax_hist_x.get_ylim()[1]*0.95, "μ(1−σ)", ha="center", va="top", fontsize=9)

        # --- Histogram: standardized domain (s) ---
        s_vals = img_std.ravel()
        ax_hist_s.hist(s_vals, bins=std_bins, density=True, color="tab:blue", alpha=0.85)
        ax_hist_s.set_xlim(std_bins[0], std_bins[-1])
        ax_hist_s.set_xlabel("standardized units (s)")
        ax_hist_s.set_ylabel("density (standardized)")
        ax_hist_s.grid(True, alpha=0.2)
        for s_ref, lab in [(0.0, "s=0"), (1.0, "s=+1"), (-1.0, "s=-1")]:
            ax_hist_s.axvline(s_ref, color="k", ls="--", lw=1.0, alpha=0.7)
            ax_hist_s.text(s_ref, ax_hist_s.get_ylim()[1]*0.95, lab, ha="center", va="top", fontsize=9)

        # Higher suptitle so it doesn't overlap anything
        fig.suptitle(
            "Standardization across detectors\n"
            "Left: images with fixed global contrast • Right: separate histograms in original vs standardized units",
            fontsize=13, y=1.06
        )
        plt.show()

    info.value = (
        f"<b>Detector:</b> {det_label} &nbsp;|&nbsp; "
        f"<b>μ</b> = {mu:.4g}, <b>σ</b> (of z = x/μ − 1) = {sig:.4g}<br>"
    )

def on_prev(_):
    if idx.value > idx.min:
        idx.value -= 1

def on_next(_):
    if idx.value < idx.max:
        idx.value += 1

def on_idx(change):
    if change.get("name") == "value":
        draw(int(change["new"]))

idx.observe(on_idx, names="value")
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)

controls = HBox([idx, btn_prev, btn_next], layout=Layout(align_items="center", gap="10px"))
display(VBox([controls, info, out]))
draw(idx.value)